In [4]:
import cv2
import numpy as np
import tensorflow as tf
import time
import requests

MODEL_PATH = "/kaggle/input/models/bubblu123/main-lrcn/tensorflow2/default/1/trial5_best.keras" 
VIDEO_PATH = "/kaggle/input/datasets/bubblu123/rob-test/Robbery047_x264.mp4"
CLASSES    = ["RoadAccidents", "Robbery", "normal"]

BOT_TOKEN = "8603086353:AAGGTwSUKoWT6y3HxuewHLMON0T9mP3IRWs"
CHAT_ID   = "6595437863"

COOLDOWN_SECONDS = 20    
def send_telegram_alert(label, conf, frame):
    message = f"🚨 *CCTV ALERT* 🚨\n\nType: {label}\nStatus: Abnormal activity detected. Check feed."
    
    temp_img = "alert_frame.jpg"
    cv2.imwrite(temp_img, cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    
    url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendPhoto"
    with open(temp_img, 'rb') as photo:
        payload = {'chat_id': CHAT_ID, 'caption': message, 'parse_mode': 'Markdown'}
        files = {'photo': photo}
        requests.post(url, data=payload, files=files)
    print(f"--- Telegram Alert Sent for {label} ---")

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
cap   = cv2.VideoCapture(VIDEO_PATH)
frame_buffer = []
last_alert_time = 0
MAX_ALERT_DURATION = 40  
first_detection_time = None
monitoring_active = True

print("Monitoring Live Feed")

while cap.isOpened() and monitoring_active:
    ret, frame = cap.read()
    if not ret: break

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (224, 224))
    frame_buffer.append(img_resized.astype("float32") / 255.0)

    if len(frame_buffer) == 40:
        input_tensor = np.expand_dims(np.stack(frame_buffer, axis=0), axis=0)
        
        preds = model.predict(input_tensor, verbose=0)[0]
        idx = np.argmax(preds)
        label = CLASSES[idx]
        conf = preds[idx]

        current_time = time.time()

        if label != "normal":
            if first_detection_time is None:
                first_detection_time = current_time
                print(f"--- FIRST DETECTION AT {time.ctime(first_detection_time)} ---")

            elapsed_time = current_time - first_detection_time
            
            if elapsed_time <= MAX_ALERT_DURATION:
                if (current_time - last_alert_time) > COOLDOWN_SECONDS:
                    send_telegram_alert(label, conf*100, img_rgb)
                    last_alert_time = current_time
            else:
                print(f"--- ALERT LIMIT REACHED ({MAX_ALERT_DURATION}s). SILENCING NOTIFICATIONS. ---")
                monitoring_active = False 

        frame_buffer.pop(0) 

cap.release()

Monitoring Live Feed
--- FIRST DETECTION AT Mon Apr  6 06:38:48 2026 ---
--- Telegram Alert Sent for Robbery ---
--- Telegram Alert Sent for Robbery ---
--- ALERT LIMIT REACHED (40s). SILENCING NOTIFICATIONS. ---
